# LegalGPT: train the deployable sentence-role ensemble (for the Review tab)

Trains the system evaluated in `paper_analysis.md` §7b on all annotated judgments:
the public-corpus InLegalBERT fine-tuned on our judgments + the BiLSTM (pretrained on the public
judgments), averaged and calibrated. 4 judgments are held out for epoch choice and calibration.

**Before you start:** L4 GPU runtime; your Hugging Face read token (the public corpus is gated and
you already accepted its terms). Run the cells in order. About 20–30 minutes.

**At the end** the model folder (~0.9 GB) is copied to Google Drive (`MyDrive/legalgpt/`). Download
that folder to the laptop and put it in `data/models/rrl/` of the project.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the package and install

In [ ]:
import os, glob, zipfile
from google.colab import files, drive
os.chdir('/content')
if not os.path.exists('/content/legaltech/backend/app'):
    up = files.upload()  # choose legalgpt_rrl.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content/legaltech')
os.chdir('/content/legaltech')
os.makedirs('data/models', exist_ok=True)
!pip -q install transformers==5.18.0 pydantic-settings==2.15.0 python-dotenv==1.2.4
!pip -q uninstall -y torchao
drive.mount('/content/drive')

## 2. The public corpus (needed to pretrain the BiLSTM)

In [ ]:
from huggingface_hub import login, hf_hub_download
login()  # paste your read token
for s in ('train', 'dev'):
    hf_hub_download('opennyaiorg/InRhetoricalRoles', f'{s}.json', repo_type='dataset', local_dir='/content/inrr')
!python backend/scripts/build_rrl_public.py --train /content/inrr/train.json --dev /content/inrr/dev.json --version v0.3

## 3. The public-corpus checkpoint (from Drive if you saved it last time, otherwise trained again, ~10 min)

In [ ]:
saved = sorted(glob.glob('/content/drive/MyDrive/legalgpt/inrr-*'))
if saved:
    CKPT = saved[-1]
    print('using the saved checkpoint', CKPT)
else:
    rc = os.system('python backend/scripts/pretrain_rrl_public.py --epochs 2 --bf16 > data/models/pretrain_public.log 2>&1')
    found = sorted(glob.glob('data/models/rrl_public/inrr-*'))
    if not found:
        os.system('tail -30 data/models/pretrain_public.log')
        raise SystemExit('stage A did not save a checkpoint; see the log above')
    CKPT = found[-1]
    os.system('grep -E "accuracy =" data/models/pretrain_public.log | tail -1')
    print('trained', CKPT)

## 4. Train the ensemble on all judgments

In [ ]:
rc = os.system(f'python backend/scripts/train_rrl_final.py --version v0.3 --stage-a {CKPT} --bf16 > data/models/final.log 2>&1')
os.system('grep -vE "Warning|UNEXPECTED|Loading weights|batch" data/models/final.log | tail -40')
OUTDIR = sorted(glob.glob('data/models/rrl/v0.3-ensemble-*'))
print('exit code', rc, '|', OUTDIR[-1] if OUTDIR else 'NO MODEL SAVED')

## 5. Copy the model to Google Drive (and download the small report)

In [ ]:
src = OUTDIR[-1]
dest = '/content/drive/MyDrive/legalgpt/' + os.path.basename(src)
os.system(f'mkdir -p /content/drive/MyDrive/legalgpt && cp -r {src} {dest} && cp data/models/final.log {dest}/')
print('copied to', dest)
!du -sh {dest}
files.download(f'{src}/metrics.json')